### 원천 데이터 분석

In [2]:
import os
from pathlib import Path
from collections import Counter
import pandas as pd

# 1. 파일명 수집 대상 경로 설정 (라벨 또는 원천데이터)
SRC_LBL_DIR = Path("../290.이안류 CCTV 데이터/01-1.정식개방데이터/Training/02.라벨링데이터")

# 2. 파일명 및 상위 폴더 수집
print("파일명 메타데이터 분석 중...")
file_records = []

# .json 파일 이름 및 폴더명 탐색
for p in SRC_LBL_DIR.rglob("*.json"):
    stem = p.stem  # 확장자 제외 파일명
    parent_dir = p.parent.name  # 상위 폴더명 (예: TL_1.JSON_1.해운대_1.GLORY)
    parts = stem.split("_")
    
    file_records.append({
        "full_name": stem,
        "folder": parent_dir,
        "parts": parts,
        "parts_len": len(parts)
    })

total_files = len(file_records)
print(f"총 분석 대상 파일 수: {total_files}개\n")

if total_files == 0:
    raise ValueError("분석할 JSON 파일이 없습니다. 경로를 확인해주세요.")

# 3. 파일명 구분자(_) 기준 분할 패턴 점검
parts_len_counts = Counter([r["parts_len"] for r in file_records])
print("=" * 60)
print("1. 언더바(_) 기준 분할된 토큰 개수 분포")
print("=" * 60)
for plen, count in parts_len_counts.items():
    print(f" - 토큰 {plen}개 구성: {count}건 ({count/total_files*100:.1f}%)")

# 4. 상위 폴더명(해수욕장/카메라 지점) 분포 확인
folder_counts = Counter([r["folder"] for r in file_records])
print("\n" + "=" * 60)
print(f"2. 상위 폴더 목록 (총 {len(folder_counts)}개 폴더/지점)")
print("=" * 60)
for folder, count in folder_counts.most_common():
    print(f" - {folder:<35} : {count:>6}개")

# 5. 토큰별 세부 의미 파악 및 집계 (가장 흔한 4개 토큰 구조 기준)
# 형식: [해수욕장코드]_[카메라위치]_[YYYYMMDD]_[HHMMSS]
parsed_data = []
for r in file_records:
    parts = r["parts"]
    if len(parts) >= 4:
        beach_code = parts[0]
        camera_id = parts[1]
        date_str = parts[2]
        time_str = parts[3]
        year = date_str[:4] if len(date_str) >= 4 else "알수없음"
        month = date_str[4:6] if len(date_str) >= 6 else "알수없음"
        hour = time_str[:2] if len(time_str) >= 2 else "알수없음"
        
        parsed_data.append({
            "beach_code": beach_code,
            "camera_id": camera_id,
            "year": year,
            "month": month,
            "hour": hour,
            "folder": r["folder"]
        })

df = pd.DataFrame(parsed_data)

if not df.empty:
    print("\n" + "=" * 60)
    print("3. 파일명 앞자리(해수욕장 코드) 분포")
    print("=" * 60)
    for code, count in df["beach_code"].value_counts().items():
        # 해당 코드가 속한 실제 폴더 이름 힌트 추출
        matched_folders = df[df["beach_code"] == code]["folder"].unique()
        folder_hint = ", ".join([f.split("_")[-2] if "해운대" in f or "송정" in f or "대천" in f else f for f in matched_folders[:2]])
        print(f" - 코드 '{code:<6}': {count:>6}건  (관련 폴더: {folder_hint})")

    print("\n" + "=" * 60)
    print("4. 카메라 지점(위치/화각 코드) 분포 Top 10")
    print("=" * 60)
    for cam, count in df["camera_id"].value_counts().head(10).items():
        print(f" - 지점 '{cam:<8}': {count:>6}건")

    print("\n" + "=" * 60)
    print("5. 연도 및 월별 분포")
    print("=" * 60)
    year_month = df.groupby(["year", "month"]).size().reset_index(name="count")
    for _, row in year_month.iterrows():
        print(f" - {row['year']}년 {row['month']}월 : {row['count']:>6}건")

    print("\n" + "=" * 60)
    print("6. 실제 파일명 샘플 10개")
    print("=" * 60)
    for r in file_records[:10]:
        print(f" - {r['full_name']} (폴더: {r['folder']})")

파일명 메타데이터 분석 중...
총 분석 대상 파일 수: 248139개

1. 언더바(_) 기준 분할된 토큰 개수 분포
 - 토큰 4개 구성: 248139건 (100.0%)

2. 상위 폴더 목록 (총 12개 폴더/지점)
 - TL_1.JSON_2.송정_2.SJHT1              :  51076개
 - TL_1.JSON_3.중문_1.BADA1              :  36759개
 - TL_1.JSON_1.해운대_2.PARA1             :  31478개
 - TL_1.JSON_3.중문_2.BADA2              :  27839개
 - TL_1.JSON_1.해운대_3.PARA2             :  21589개
 - TL_1.JSON_1.해운대_1.GLORY             :  20965개
 - TL_1.JSON_1.해운대_4.SEAC1             :  14138개
 - TL_1.JSON_2.송정_1.WHIB1              :  12860개
 - TL_1.JSON_5.낙산_2.NSBE2              :  10641개
 - TL_1.JSON_4.대천_1.MUDCH              :   8536개
 - TL_1.JSON_5.낙산_1.NSBE1              :   8504개
 - TL_1.JSON_4.대천_2.ZIPTR              :   3754개

3. 파일명 앞자리(해수욕장 코드) 분포
 - 코드 'HD    ':  88170건  (관련 폴더: 1.해운대, 1.해운대)
 - 코드 'JM    ':  64598건  (관련 폴더: TL_1.JSON_3.중문_1.BADA1, TL_1.JSON_3.중문_2.BADA2)
 - 코드 'SJ    ':  63936건  (관련 폴더: 2.송정, 2.송정)
 - 코드 'NS    ':  19145건  (관련 폴더: TL_1.JSON_5.낙산_1.NSBE1, TL_1.JSON_5.낙산_2.NSBE2)
 - 코드 'DC 

### 원천 데이터 1차 분리

In [ ]:
import json
import os
import shutil
from pathlib import Path
from PIL import Image

# 1. 디렉토리 설정
BASE_DIR = Path("290.이안류 CCTV 데이터/01-1.정식개방데이터/Training")
SRC_IMG_DIR = BASE_DIR / "01.원천데이터"
SRC_LBL_DIR = BASE_DIR / "02.라벨링데이터"

TARGET_TRAIN_IMG = Path("dataset_original/train/images")
TARGET_TRAIN_LBL = Path("dataset_original/train/labels")

TARGET_TRAIN_IMG.mkdir(parents=True, exist_ok=True)
TARGET_TRAIN_LBL.mkdir(parents=True, exist_ok=True)

# 2. 원천 이미지 사전 인덱싱 (폴더명 Prefix 불일치 완벽 해결)
print("1단계: 원천 이미지 인덱싱 중...")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG"}

image_lookup = {}
for p in SRC_IMG_DIR.rglob("*"):
    if p.suffix in IMAGE_EXTENSIONS:
        image_lookup[p.stem] = p

print(f"-> 총 {len(image_lookup)}개의 원천 이미지 색인 완료.")

# 3. 라벨 파일 변환 및 누락된 이미지만 복사
label_files = list(SRC_LBL_DIR.rglob("*.json"))
print(f"2단계: 총 {len(label_files)}개의 라벨 처리 시작...\n")

copied_img_count = 0
skipped_img_count = 0
processed_lbl_count = 0

for idx, lbl_path in enumerate(label_files):
    stem = lbl_path.stem
    img_path = image_lookup.get(stem)

    if not img_path:
        continue

    folder_prefix = img_path.parent.name
    dst_img_name = f"{folder_prefix}_{img_path.name}"
    dst_lbl_name = f"{folder_prefix}_{stem}.txt"

    dst_img_path = TARGET_TRAIN_IMG / dst_img_name
    dst_lbl_path = TARGET_TRAIN_LBL / dst_lbl_name

    # [핵심] 이미지가 이미 존재하면 복사하지 않고 패스 (시간 절약)
    if not dst_img_path.exists():
        shutil.copy2(img_path, dst_img_path)
        copied_img_count += 1
    else:
        skipped_img_count += 1

    # 라벨 JSON 파싱 및 정밀 변환
    with open(lbl_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    annotations = data.get("annotations", {})
    cls_val = annotations.get("class", 0)

    # Case 1: class == 0 (배경 이미지)
    if cls_val == 0:
        open(dst_lbl_path, "w").close()
        processed_lbl_count += 1
        continue

    # Case 2: class == 1 (이안류 검출)
    if cls_val == 1:
        raw_drawing = annotations.get("drawing", [])
        if not raw_drawing:
            open(dst_lbl_path, "w").close()
            processed_lbl_count += 1
            continue

        # 원본 이미지 해상도 조회
        with Image.open(img_path) as img:
            img_w, img_h = img.size

        yolo_lines = []
        cls_id = 0

        for box in raw_drawing:
            if not isinstance(box, list) or len(box) < 4:
                continue

            p1, p2, p3, p4 = box[0], box[1], box[2], box[3]
            xs = [p1[0], p2[0], p3[0], p4[0]]
            ys = [p1[1], p2[1], p3[1], p4[1]]

            min_x, max_x = min(xs), max(xs)
            min_y, max_y = min(ys), max(ys)

            x_center = ((min_x + max_x) / 2.0) / img_w
            y_center = ((min_y + max_y) / 2.0) / img_h
            w = (max_x - min_x) / img_w
            h = (max_y - min_y) / img_h

            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            w = max(0.0, min(1.0, w))
            h = max(0.0, min(1.0, h))

            yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

        with open(dst_lbl_path, "w", encoding="utf-8") as out_f:
            out_f.write("\n".join(yolo_lines))

        processed_lbl_count += 1

    if (idx + 1) % 10000 == 0:
        print(f"진행 상황: {idx + 1} / {len(label_files)} 완료...")

print(f"\n최종 완료!")
print(f"- 기존에 있어 건너뛴 이미지 수 (시간 절약): {skipped_img_count}건")
print(f"- 새로 추가 복사된 이미지 수: {copied_img_count}건")
print(f"- 정상 생성/갱신된 라벨 수: {processed_lbl_count}건")

### 원천 데이터 2차 분리

In [ ]:
import os
import shutil
import random
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# 1. 경로 설정
SRC_DIR = Path("dataset_original/train")
SRC_IMG = SRC_DIR / "images"
SRC_LBL = SRC_DIR / "labels"

TARGET_BASE = Path("dataset_integrated_v2")

TARGET_COUNTS = {
    "train": {"pos": 8000, "neg": 1200},
    "valid": {"pos": 1000, "neg": 200},
    "test":  {"pos": 500,  "neg": 100}
}

# 2. 타깃 폴더 생성
for split in ["train", "valid", "test"]:
    (TARGET_BASE / split / "images").mkdir(parents=True, exist_ok=True)
    (TARGET_BASE / split / "labels").mkdir(parents=True, exist_ok=True)

# 3. 초고속 사전 인덱싱 (os.scandir 활용 - 5초 소요)
print("1단계: 이미지 및 라벨 초고속 색인 중 (5~10초 소요)...")

img_dict = {}
with os.scandir(SRC_IMG) as it:
    for entry in it:
        if entry.is_file():
            # stem 추출 (확장자 제거)
            stem = os.path.splitext(entry.name)[0]
            img_dict[stem] = entry.path

lbl_dict = {}
with os.scandir(SRC_LBL) as it:
    for entry in it:
        if entry.is_file() and entry.name.endswith(".txt"):
            stem = os.path.splitext(entry.name)[0]
            # stat()을 바로 가져와 파일 크기 확인 (I/O 병목 제거)
            lbl_dict[stem] = (entry.path, entry.stat().st_size)

print(f" -> 색인 완료: 이미지 {len(img_dict)}개, 라벨 {len(lbl_dict)}개")

# 4. 공통 매칭 데이터 메타데이터 파싱
print("\n2단계: 매칭 데이터 분석 및 메타데이터 파싱...")
records = []
common_stems = set(img_dict.keys()) & set(lbl_dict.keys())

for stem in common_stems:
    img_path = img_dict[stem]
    lbl_path, lbl_size = lbl_dict[stem]

    # 파일명 파싱 (파일명 끝자리 기준: BEACH_CAM_YYYYMMDD_HHMMSS)
    parts = stem.split("_")
    if len(parts) >= 4:
        date_str = parts[-2]
        time_str = parts[-1]
        cam = parts[-3]
        beach = parts[-4]
        try:
            dt = datetime.strptime(f"{date_str}_{time_str}", "%Y%m%d_%H%M%S")
        except ValueError:
            dt = datetime.min
    else:
        beach, cam, date_str, time_str = "UNKNOWN", "UNKNOWN", "UNKNOWN", "UNKNOWN"
        dt = datetime.min

    records.append({
        "stem": stem,
        "img_path": img_path,
        "lbl_path": lbl_path,
        "beach": beach,
        "cam": cam,
        "date": date_str,
        "datetime": dt,
        "is_pos": (lbl_size > 0)
    })

print(f" -> 총 매칭 유효 데이터: {len(records)}건")

# 5. 시간 필터링 (최소 15초 간격)
print("\n3단계: 초 단위 연사 중복 제거 (최소 15초 텀 필터링)...")
grouped_by_cam_date = defaultdict(list)
for r in records:
    group_key = f"{r['beach']}_{r['cam']}_{r['date']}"
    grouped_by_cam_date[group_key].append(r)

filtered_records = []
MIN_INTERVAL_SECONDS = 15

for g_key, recs in grouped_by_cam_date.items():
    recs.sort(key=lambda x: x["datetime"])
    last_dt = None
    for r in recs:
        if last_dt is None or (r["datetime"] - last_dt).total_seconds() >= MIN_INTERVAL_SECONDS:
            filtered_records.append(r)
            last_dt = r["datetime"]

print(f" -> 15초 필터링 통과 데이터: {len(filtered_records)}건")

# 6. 해수욕장 도메인 격리 (Train/Val vs Test)
test_candidates = [r for r in filtered_records if r["beach"] in ["NS", "DC"]]
train_val_candidates = [r for r in filtered_records if r["beach"] in ["HD", "SJ", "JM"]]

print(f"\n4단계: 도메인 격리 현황")
print(f" - Train/Val 대상군 (HD, SJ, JM): {len(train_val_candidates)}건")
print(f" - Test 격리군 (NS, DC): {len(test_candidates)}건")

# Test 세트 추출
test_pos = [r for r in test_candidates if r["is_pos"]]
test_neg = [r for r in test_candidates if not r["is_pos"]]
random.seed(42)
random.shuffle(test_pos)
random.shuffle(test_neg)

selected_test = (
    test_pos[:TARGET_COUNTS["test"]["pos"]] + 
    test_neg[:TARGET_COUNTS["test"]["neg"]]
)

# 7. Train / Valid 날짜(Date) 기반 분할
available_dates = sorted(list({r["date"] for r in train_val_candidates if r["date"] != "UNKNOWN"}))
random.seed(42)
random.shuffle(available_dates)

date_stats = defaultdict(lambda: {"pos": [], "neg": []})
for r in train_val_candidates:
    if r["is_pos"]:
        date_stats[r["date"]]["pos"].append(r)
    else:
        date_stats[r["date"]]["neg"].append(r)

val_dates = set()
val_pos_count = 0
val_neg_count = 0
target_val_pos = TARGET_COUNTS["valid"]["pos"]
target_val_neg = TARGET_COUNTS["valid"]["neg"]

for d in available_dates:
    if val_pos_count < target_val_pos or val_neg_count < target_val_neg:
        val_dates.add(d)
        val_pos_count += len(date_stats[d]["pos"])
        val_neg_count += len(date_stats[d]["neg"])
    else:
        break

train_dates = set(available_dates) - val_dates

val_pool_pos = [r for r in train_val_candidates if r["date"] in val_dates and r["is_pos"]]
val_pool_neg = [r for r in train_val_candidates if r["date"] in val_dates and not r["is_pos"]]

train_pool_pos = [r for r in train_val_candidates if r["date"] in train_dates and r["is_pos"]]
train_pool_neg = [r for r in train_val_candidates if r["date"] in train_dates and not r["is_pos"]]

random.shuffle(val_pool_pos)
random.shuffle(val_pool_neg)
random.shuffle(train_pool_pos)
random.shuffle(train_pool_neg)

selected_val = (
    val_pool_pos[:TARGET_COUNTS["valid"]["pos"]] + 
    val_pool_neg[:TARGET_COUNTS["valid"]["neg"]]
)

selected_train = (
    train_pool_pos[:TARGET_COUNTS["train"]["pos"]] + 
    train_pool_neg[:TARGET_COUNTS["train"]["neg"]]
)

# 8. dataset_obj 파일 복사 실행
def copy_records(record_list, split_name):
    dst_img_dir = TARGET_BASE / split_name / "images"
    dst_lbl_dir = TARGET_BASE / split_name / "labels"
    
    pos_cnt = sum(1 for r in record_list if r["is_pos"])
    neg_cnt = len(record_list) - pos_cnt

    for r in record_list:
        shutil.copy2(r["img_path"], dst_img_dir / os.path.basename(r["img_path"]))
        shutil.copy2(r["lbl_path"], dst_lbl_dir / os.path.basename(r["lbl_path"]))

    print(f"[{split_name.upper()}] 복사 완료 -> 이안류: {pos_cnt}장, 배경: {neg_cnt}장 (합계: {len(record_list)}장)")

print("\n5단계: dataset_obj로 최종 파일 복사 진행...")
copy_records(selected_train, "train")
copy_records(selected_val, "valid")
copy_records(selected_test, "test")

print("\n🎉 모든 분할 및 복사 작업이 완료되었습니다!")

### 스페인 해변 데이터 정제

In [ ]:
import os
import shutil
from pathlib import Path
from collections import defaultdict

# 1. 경로 설정
SRC_BASE = Path("selected_balanced_8to2")
SRC_IMG_DIR = SRC_BASE / "images"
SRC_LBL_DIR = SRC_BASE / "labels"

# 새로 분류할 폴더 (기존에 뭉쳐진 폴더가 있다면 덮어쓰거나 구분되도록 설정)
DST_BASE = Path("beaches_classified_v2")

assert SRC_IMG_DIR.exists(), f"❌ 이미지 폴더가 없습니다: {SRC_IMG_DIR.resolve()}"
assert SRC_LBL_DIR.exists(), f"❌ 라벨 폴더가 없습니다: {SRC_LBL_DIR.resolve()}"

# 2. 해변 식별자 추출 함수
def extract_beach_id(filename: str) -> str:
    # 1) PHOTO 로 시작하는 경우 단일 해변으로 처리
    if filename.startswith("PHOTO"):
        return "해변_PHOTO"
    
    # 2) 언더스코어(_) 기준으로 분할
    tokens = filename.split("_")
    
    # clm_s_01_xxxx 형태: 앞의 3개 블록('clm_s_01')을 고유 해변 ID로 결합
    if len(tokens) >= 3 and (tokens[0].isalpha() or tokens[0].isalnum()):
        # 만약 세 번째 토큰이 2~3자리 숫자/문자 식별자(01, 02, 03 등)인 경우
        return f"해변_{tokens[0]}_{tokens[1]}_{tokens[2]}"
    elif len(tokens) >= 2:
        return f"해변_{tokens[0]}_{tokens[1]}"
    else:
        # 언더스코어가 없는 경우 앞 3글자
        return f"해변_{filename[:3]}"

# 3. 대상 이미지 스캔
img_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
img_files = [p for p in SRC_IMG_DIR.iterdir() if p.suffix.lower() in img_extensions]

print("=" * 75)
print(f"📦 분류 대상 총 이미지: {len(img_files):,} 장")
print(f"📁 새 결과 저장 경로: {DST_BASE.resolve()}")
print("=" * 75)

stats = defaultdict(lambda: {"images": 0, "labels": 0, "backgrounds": 0})

# 4. 해변별 디렉토리 생성 및 복사/분류
for img_path in img_files:
    stem = img_path.stem
    beach_id = extract_beach_id(stem)
    
    # 해변별 저장 경로 (beaches_classified_v2/해변_clm_s_01/images, labels)
    target_img_dir = DST_BASE / beach_id / "images"
    target_lbl_dir = DST_BASE / beach_id / "labels"
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    # 1) 이미지 복사
    shutil.copy2(img_path, target_img_dir / img_path.name)
    stats[beach_id]["images"] += 1
    
    # 2) 라벨 파일 확인 및 복사
    lbl_path = SRC_LBL_DIR / f"{stem}.txt"
    if lbl_path.exists():
        shutil.copy2(lbl_path, target_lbl_dir / lbl_path.name)
        if lbl_path.stat().st_size > 0:
            stats[beach_id]["labels"] += 1
        else:
            stats[beach_id]["backgrounds"] += 1
    else:
        # 라벨이 없는 경우 빈 텍스트(배경 이미지용) 생성
        (target_lbl_dir / f"{stem}.txt").touch()
        stats[beach_id]["backgrounds"] += 1

# 5. 분류 결과 리포트 출력
print(f"{'해변(도메인 식별자)':<25} | {'이미지 수':<10} | {'양성 라벨':<10} | {'배경(Negative)':<14}")
print("-" * 75)

total_imgs = 0
total_pos = 0
total_bg = 0

for beach_id, count in sorted(stats.items(), key=lambda x: x[1]["images"], reverse=True):
    total_imgs += count["images"]
    total_pos += count["labels"]
    total_bg += count["backgrounds"]
    bg_ratio = (count["backgrounds"] / count["images"]) * 100 if count["images"] > 0 else 0.0
    print(f"{beach_id:<25} | {count['images']:<10,} | {count['labels']:<10,} | {count['backgrounds']:<5,} ({bg_ratio:4.1f}%)")

print("-" * 75)
total_bg_ratio = (total_bg / total_imgs) * 100 if total_imgs > 0 else 0.0
print(f"{'합계':<25} | {total_imgs:<10,} | {total_pos:<10,} | {total_bg:<5,} ({total_bg_ratio:4.1f}%)")
print("=" * 75)
print(f"✅ '{DST_BASE.name}' 폴더에 clm_s_01, clm_s_03 등 세부 해변별로 정상 분리되었습니다.")